In [ ]:
# Model Evaluation

This notebook evaluates the trained CNN model using the test dataset.
Instead of relying solely on accuracy, we analyze:

- Precision
- Recall
- F1-score
- Confusion Matrix
- Threshold sensitivity

This analysis explains the observed model behavior and limitations.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

BASE_DIR = "data"
TEST_DIR = os.path.join(BASE_DIR, "test", "images")
MODEL_PATH = "models/cnn_chart_model.keras"


In [ ]:
model = load_model(MODEL_PATH)

test_gen = ImageDataGenerator(rescale=1./255)

test_data = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_data.class_indices


In [ ]:
y_prob = model.predict(test_data).flatten()
y_true = test_data.classes

print("Probability stats:")
print("min:", y_prob.min())
print("max:", y_prob.max())
print("mean:", y_prob.mean())


In [ ]:
threshold = 0.5
y_pred = (y_prob > threshold).astype(int)

print("Threshold:", threshold)
print(classification_report(y_true, y_pred, target_names=["down", "up"]))


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["down", "up"],
            yticklabels=["down", "up"])
plt.title("Confusion Matrix (Threshold = 0.5)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
thresholds = [0.4, 0.45, 0.5, 0.55, 0.6]

for t in thresholds:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold = {t}")
    print(classification_report(y_true, y_pred, target_names=["down", "up"]))


In [ ]:
## Evaluation Conclusions

- The model outputs probabilities clustered around 0.5, indicating low confidence.
- Small changes in the classification threshold drastically alter predictions.
- The CNN collapses to predicting a single class under most thresholds.
- Accuracy is misleading due to class imbalance.

Overall, this project demonstrates that detecting technical patterns from raw
financial chart images is a highly challenging task and may require:
- More informative representations
- Temporal context
- Non-image-based features
